# LangGraph Sequential Workflows

**VidTrace course reconstruction — Agentic AI Class, 17 Aug 2026**

This notebook follows the extracted lecture structure:
1. Non-LLM sequential workflow — BMI Calculator
2. LLM sequential workflow
3. Prompt-chaining sequential workflow

The extraction shows the BMI state as `weight`, `height`, and `bmi`, and the prompt-chaining example as `topic → outline → content` using **AI for Healthcare**.

> **Reconstruction note:** the notebook is designed to be runnable in Colab. The lecture's screen was captured through OCR, so code shown here is a clean runnable reconstruction of the extracted workflow rather than a claim that every character is verbatim from the recording.

In [ ]:
!pip -q install -U langgraph langchain-core

from typing import TypedDict
from pprint import pprint
from langgraph.graph import StateGraph, START, END

print("LangGraph setup complete.")

## 1. Non-LLM Sequential Workflow — BMI Calculator

The extracted lecture describes three nodes:

`Input Node → Calculate BMI → Output Node`

The visible example uses:
- weight = 70
- height = 1.75
- bmi initially = None
- final bmi = 22.86

In [ ]:
class BMIState(TypedDict):
    weight: float
    height: float
    bmi: float | None

def input_node(state: BMIState):
    return state

def calculate_bmi(state: BMIState):
    bmi = state["weight"] / (state["height"] ** 2)
    return {"bmi": round(bmi, 2)}

def output_node(state: BMIState):
    print("Final BMI:", state["bmi"])
    return state

builder = StateGraph(BMIState)
builder.add_node("input", input_node)
builder.add_node("calculate_bmi", calculate_bmi)
builder.add_node("output", output_node)

builder.add_edge(START, "input")
builder.add_edge("input", "calculate_bmi")
builder.add_edge("calculate_bmi", "output")
builder.add_edge("output", END)

bmi_graph = builder.compile()

result = bmi_graph.invoke({
    "weight": 70,
    "height": 1.75,
    "bmi": None
})

pprint(result)

### Observe the state flow

The important idea from the lecture is that each node reads the current state and the next node receives the updated state. The Input/Output nodes are mainly used to start and terminate the workflow, while the calculation happens in the BMI node.

In [ ]:
# Inspect the state after execution
assert round(result["bmi"], 2) == 22.86
print(result)

## 2. LLM Sequential Workflow

The lecture explains the sequential LLM pattern as:

`LLM 1 output → input/prompt for LLM 2`

To keep this notebook runnable without an API key, we use a deterministic stand-in with the same state-transition shape.

In [ ]:
class LLMState(TypedDict):
    question: str
    answer: str | None

class DemoLLM:
    def invoke(self, prompt: str) -> str:
        return f"Demo LLM response for: {prompt}"

llm = DemoLLM()

def ask_llm(state: LLMState):
    return {"answer": llm.invoke(state["question"])}

b = StateGraph(LLMState)
b.add_node("ask_llm", ask_llm)
b.add_edge(START, "ask_llm")
b.add_edge("ask_llm", END)

llm_graph = b.compile()

pprint(llm_graph.invoke({
    "question": "Explain agentic workflows in simple terms.",
    "answer": None
}))

## 3. Prompt-Chaining Sequential Workflow

The extracted slide shows:

`Topic → LLM: Create Outline → LLM: Write Blog`

State fields:
- `topic`
- `outline`
- `content`

Example topic: **AI for Healthcare**.

In [ ]:
class PromptChainState(TypedDict):
    topic: str
    outline: str | None
    content: str | None

def topic_node(state: PromptChainState):
    return state

def create_outline(state: PromptChainState):
    topic = state["topic"]
    outline = (
        f"1. Introduction to {topic}\n"
        "2. Use Cases\n"
        "3. Challenges"
    )
    return {"outline": outline}

def write_blog(state: PromptChainState):
    content = (
        f"Topic: {state['topic']}\n\n"
        f"Outline:\n{state['outline']}\n\n"
        "Draft content generated from the outline."
    )
    return {"content": content}

p = StateGraph(PromptChainState)
p.add_node("topic", topic_node)
p.add_node("create_outline", create_outline)
p.add_node("write_blog", write_blog)

p.add_edge(START, "topic")
p.add_edge("topic", "create_outline")
p.add_edge("create_outline", "write_blog")
p.add_edge("write_blog", END)

prompt_chain = p.compile()

final = prompt_chain.invoke({
    "topic": "AI for Healthcare",
    "outline": None,
    "content": None
})

pprint(final)

## Key concepts from the lecture

- A **state** carries information through the workflow.
- A **sequential edge** means the downstream node waits for the previous node.
- In prompt chaining, one LLM step can create structured information that becomes the input to the next step.
- The extracted lecture places the BMI examples around the early part of the recording and the prompt-chaining example around the later part of the 17 Aug class.